In [1]:
!pip install ultralytics -q
!pip install huggingface_hub -q

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"CUDA: {torch.cuda.is_available()}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.0 MB/s eta 0:00:00a 0:00:01
GPU: Tesla T4
CUDA: True


In [2]:
import torch
from pathlib import Path

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"CUDA: {torch.cuda.is_available()}")

# Check what's mounted
print("\nMounted datasets:")
for p in Path("/kaggle/input").iterdir():
    print(f"  {p.name}/")
    for sub in p.iterdir():
        files = list(sub.rglob("*.*"))
        print(f"    {sub.name}/: {len(files)} files")

GPU: Tesla T4
CUDA: True

Mounted datasets:
  datasets/
    apoorv007/: 9614 files


In [3]:
from pathlib import Path

base = Path("/kaggle/input/datasets/apoorv007")

# Find all files
all_imgs = list(base.rglob("*.png")) + list(base.rglob("*.jpg"))
all_lbls = list(base.rglob("*.txt"))
all_pts  = list(Path("/kaggle/input").rglob("*.pt"))

t_imgs = [f for f in all_imgs if f.stem.startswith('t-')]
v_imgs = [f for f in all_imgs if f.stem.startswith('v-')]
t_lbls = [f for f in all_lbls if f.stem.startswith('t-')]
v_lbls = [f for f in all_lbls if f.stem.startswith('v-')]

print(f"Train images : {len(t_imgs)}")
print(f"Val   images : {len(v_imgs)}")
print(f"Train labels : {len(t_lbls)}")
print(f"Val   labels : {len(v_lbls)}")
print(f"Checkpoints  : {all_pts}")

# Save for next cells
TRAIN_IMGS = t_imgs
VAL_IMGS   = v_imgs
LBL_INDEX  = {f.stem: f for f in all_lbls}
BEST_PT    = all_pts[0] if all_pts else None

print(f"\nBest.pt path : {BEST_PT}")
print(f"\nSample train : {t_imgs[0] if t_imgs else 'NONE'}")
print(f"Sample val   : {v_imgs[0] if v_imgs else 'NONE'}")
print(f"Sample label : {t_lbls[0] if t_lbls else 'NONE'}")

Train images : 3613
Val   images : 1194
Train labels : 3613
Val   labels : 1194
Checkpoints  : []

Best.pt path : None

Sample train : /kaggle/input/datasets/apoorv007/glh-bridge-full/glh-bridge-full/images/images/train/t-2919.png
Sample val   : /kaggle/input/datasets/apoorv007/glh-bridge-full/glh-bridge-full/images/images/val/v-1065.png
Sample label : /kaggle/input/datasets/apoorv007/glh-bridge-full/glh-bridge-full/labelTxt/labelTxt/train/t-543.txt


In [4]:
import os, cv2
import numpy as np
from PIL import Image
from pathlib import Path
from tqdm import tqdm

IMG_TRAIN = Path("/kaggle/input/datasets/apoorv007/glh-bridge-full/glh-bridge-full/images/images/train")
IMG_VAL   = Path("/kaggle/input/datasets/apoorv007/glh-bridge-full/glh-bridge-full/images/images/val")
LBL_TRAIN = Path("/kaggle/input/datasets/apoorv007/glh-bridge-full/glh-bridge-full/labelTxt/labelTxt/train")
LBL_VAL   = Path("/kaggle/input/datasets/apoorv007/glh-bridge-full/glh-bridge-full/labelTxt/labelTxt/val")
BEST_PT   = "/kaggle/input/datasets/apoorv007/bridge-checkpoint/best.pt"

# Only labels go to working dir
for split in ['train','val']:
    os.makedirs(f"/kaggle/working/dataset/labels/{split}", exist_ok=True)

def is_water(img_np, cx, cy, w, h):
    H, W = img_np.shape[:2]
    pad  = 25
    x1   = max(0, int((cx-w/2)*W)-pad)
    y1   = max(0, int((cy-h/2)*H)-pad)
    x2   = min(W, int((cx+w/2)*W)+pad)
    y2   = min(H, int((cy+h/2)*H)+pad)
    region = img_np[y1:y2, x1:x2].reshape(-1,3)
    if len(region) == 0: return False
    if len(region) > 400:
        region = region[np.random.choice(len(region), 400, replace=False)]
    water = 0
    for p in region:
        r,g,b  = int(p[0]),int(p[1]),int(p[2])
        bright = (r+g+b)/3
        if b>r and b>g and bright<130:                    water+=1
        elif abs(r-g)<25 and abs(g-b)<25 and bright<100: water+=1
        elif b>90 and g>80 and r<120 and b>r:             water+=1
    return (water/len(region)) >= 0.28

def process(img_dir, lbl_dir, split):
    imgs = list(img_dir.glob("*.png")) + list(img_dir.glob("*.jpg"))
    kept, total_b, water_b = 0, 0, 0

    pbar = tqdm(imgs, desc=f"  {split}", unit="img",
                bar_format="{l_bar}{bar:30}{r_bar}")

    for img_path in pbar:
        lbl_path = lbl_dir / (img_path.stem + ".txt")
        if not lbl_path.exists(): continue
        try:
            img_pil = Image.open(img_path)
            W, H    = img_pil.size
            img_np  = cv2.cvtColor(
                          cv2.imread(str(img_path)),
                          cv2.COLOR_BGR2RGB)
        except: continue

        water_lines = []
        with open(lbl_path) as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('imagesource') \
                   or line.startswith('gsd'): continue
                parts = line.split()
                if len(parts) < 9: continue
                try:
                    coords  = list(map(float, parts[:8]))
                    xs, ys  = coords[0::2], coords[1::2]
                    x1, x2  = min(xs), max(xs)
                    y1, y2  = min(ys), max(ys)
                    cx = ((x1+x2)/2)/W
                    cy = ((y1+y2)/2)/H
                    w  = (x2-x1)/W
                    h  = (y2-y1)/H
                    cx,cy = max(0,min(1,cx)), max(0,min(1,cy))
                    w,h   = max(0.001,min(1,w)), max(0.001,min(1,h))
                    total_b += 1
                    if is_water(img_np, cx, cy, w, h):
                        water_lines.append(
                            f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
                        water_b += 1
                except: continue

        if water_lines:
            with open(
                f"/kaggle/working/dataset/labels/{split}/{img_path.stem}.txt",
                'w') as f:
                f.write('\n'.join(water_lines))
            kept += 1

        # Update progress bar with live stats
        pbar.set_postfix({
            'water': water_b,
            'kept' : kept,
            'total': total_b
        })

    print(f"\n✓ {split}: {water_b}/{total_b} water bridges → {kept} images kept")
    return kept

print("="*50)
print(" Water Bridge Filtering")
print("="*50)
print("\nProcessing train...")
process(IMG_TRAIN, LBL_TRAIN, 'train')
print("\nProcessing val...")
process(IMG_VAL, LBL_VAL, 'val')
print("\n✓ Done! Labels saved to /kaggle/working/dataset/labels/")

 Water Bridge Filtering

Processing train...


  train:   1%|▎                             | 31/3613 [00:07<12:20,  4.84img/s, water=220, kept=29, total=277]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (163840000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
  train: 100%|██████████████████████████████| 3613/3613 [18:09<00:00,  3.32img/s, water=26460, kept=3463, total=32873]



✓ train: 26460/32873 water bridges → 3463 images kept

Processing val...


  val: 100%|██████████████████████████████| 1194/1194 [05:59<00:00,  3.32img/s, water=8995, kept=1145, total=11722]


✓ val: 8995/11722 water bridges → 1145 images kept

✓ Done! Labels saved to /kaggle/working/dataset/labels/


In [ ]:
import yaml, os
from pathlib import Path
from ultralytics import YOLO
import torch, gc, os

tl = len(list(Path('/kaggle/working/dataset/labels/train').glob('*.txt')))
vl = len(list(Path('/kaggle/working/dataset/labels/val').glob('*.txt')))
print(f"Train labels: {tl}")
print(f"Val   labels: {vl}")

# Symlink images into working dir (reverse — link images, keep labels)
# Working dir has space for symlinks (they're just pointers, no actual data copied)
os.makedirs("/kaggle/working/dataset/images/train", exist_ok=True)
os.makedirs("/kaggle/working/dataset/images/val",   exist_ok=True)

IMG_TRAIN = "/kaggle/input/datasets/apoorv007/glh-bridge-full/glh-bridge-full/images/images/train"
IMG_VAL   = "/kaggle/input/datasets/apoorv007/glh-bridge-full/glh-bridge-full/images/images/val"

# Symlink each image into working/images/ (no data copied — just pointers)
from pathlib import Path
from tqdm import tqdm

for split, src in [('train', IMG_TRAIN), ('val', IMG_VAL)]:
    imgs = list(Path(src).glob("*.png")) + list(Path(src).glob("*.jpg"))
    dst  = Path(f"/kaggle/working/dataset/images/{split}")
    # Only symlink images that have water labels
    lbls = set(f.stem for f in Path(f"/kaggle/working/dataset/labels/{split}").glob("*.txt"))
    
    linked = 0
    for img in tqdm(imgs, desc=f"Linking {split}"):
        if img.stem in lbls:
            link = dst / img.name
            if not link.exists():
                link.symlink_to(img)
            linked += 1
    print(f"✓ {split}: {linked} image symlinks created")

# Check disk (symlinks use almost no space)
import shutil as sh
_, _, free = sh.disk_usage("/kaggle/working")
print(f"\nFree disk: {free / (1024**3):.1f} GB")

# YAML — now everything is under /kaggle/working/
cfg = {
    'path':  '/kaggle/working/dataset',
    'train': 'images/train',
    'val':   'images/val',
    'nc':    1,
    'names': ['bridge']
}
with open('water_bridge.yaml','w') as f:
    yaml.dump(cfg, f)

print("YAML saved!")

torch.cuda.empty_cache()
gc.collect()
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

print(f"GPU free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated())/1e9:.1f} GB")


model = YOLO('yolov8s.pt')

model.train(
    data='water_bridge.yaml',
    epochs=100,
    imgsz=640,
    batch=8,
    name='water_bridge_FINAL',
    patience=20,
    dropout=0.2,
    weight_decay=0.0005,
    lr0=0.01,
    lrf=0.01,
    warmup_epochs=3,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    flipud=0.3,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    degrees=15.0,
    translate=0.1,
    scale=0.5,
    save=True,
    save_period=5,
    plots=True,
    verbose=True
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Train labels: 3463
Val   labels: 1145


Linking train: 100%|██████████| 3613/3613 [00:00<00:00, 21235.33it/s]


✓ train: 3463 image symlinks created


Linking val: 100%|██████████| 1194/1194 [00:00<00:00, 24708.16it/s]


✓ val: 1145 image symlinks created

Free disk: 19.5 GB
YAML saved!
GPU free: 15.6 GB
Ultralytics 8.4.39 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=water_bridge.yaml, degrees=15.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.2, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.3, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=water_bridge_FINAL, nbs=64, 

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (163840000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


train: Scanning /kaggle/working/dataset/labels/train... 3463 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3463/3463 164.3it/s 21.1s<0.0s
train: /kaggle/working/dataset/images/train/t-2496.png: 1 duplicate labels removed
train: /kaggle/working/dataset/images/train/t-2567.png: 1 duplicate labels removed
train: /kaggle/working/dataset/images/train/t-3543.png: 2 duplicate labels removed
train: /kaggle/working/dataset/images/train/t-3601.png: 1 duplicate labels removed
train: /kaggle/working/dataset/images/train/t-690.png: 1 duplicate labels removed
train: New cache created: /kaggle/working/dataset/labels/train.cache
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1000.9±736.0 MB/s, size: 3489.3 KB)
val: Scanning /kaggle/working/dataset/labels/val... 201 images, 0 backgro

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (163840000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


val: Scanning /kaggle/working/dataset/labels/val... 1145 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1145/1145 181.6it/s 6.3s0.2s
val: /kaggle/working/dataset/images/val/v-69.png: 1 duplicate labels removed
val: New cache created: /kaggle/working/dataset/labels/val.cache
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
Plotting labels to /kaggle/working/runs/detect/water_bridge_FINAL/labels.jpg... 
Image sizes 640 train, 640 val
Using 2 dataloader workers
Logging results to /kaggle/working/runs/detect/water_bridge_FINAL
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/100      2.12G      3.119      4.178      1.389         74        640: 100% ━━━━━━━━━━━━ 433/433 1.3s/it 9:02<0.4s

In [ ]:
from ultralytics import YOLO

model = YOLO('runs/detect/water_bridge_s/weights/best.pt')
metrics = model.val(data='water_bridge.yaml')

print(f"\n{'='*40}")
print(f"  mAP@0.50      : {metrics.box.map50:.4f}")
print(f"  mAP@0.50:0.95 : {metrics.box.map:.4f}")
print(f"  Precision     : {metrics.box.mp:.4f}")
print(f"  Recall        : {metrics.box.mr:.4f}")
print(f"{'='*40}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob, random
from pathlib import Path
from ultralytics import YOLO

model = YOLO('runs/detect/water_bridge_s/weights/best.pt')

val_imgs = glob.glob("/kaggle/working/dataset/images/val/*.png")
sample   = random.sample(val_imgs, min(9, len(val_imgs)))

model.predict(
    source=sample,
    save=True,
    conf=0.25,
    project="/kaggle/working/outputs",
    name="predictions"
)

pred_imgs = sorted(glob.glob("/kaggle/working/outputs/predictions/*.png"))[:9]

fig, axes = plt.subplots(3, 3, figsize=(18, 18))
fig.suptitle("YOLOv8s — Bridge Over Water Detection\nGreen boxes = detected bridges",
             fontsize=14, fontweight='bold')

for ax, p in zip(axes.flatten(), pred_imgs):
    ax.imshow(mpimg.imread(p))
    ax.axis('off')
    ax.set_title(Path(p).stem, fontsize=8)

plt.tight_layout()
plt.savefig("/kaggle/working/outputs/results_grid.png", dpi=150)
plt.show()
print("Saved: /kaggle/working/outputs/results_grid.png")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

results_dir = Path("runs/detect/water_bridge_s")

# YOLOv8 auto-saves these plots
plots = {
    'results.png'         : 'Loss & mAP curves',
    'confusion_matrix.png': 'Confusion Matrix',
    'PR_curve.png'        : 'Precision-Recall Curve',
    'F1_curve.png'        : 'F1 Curve',
    'val_batch0_pred.png' : 'Sample Val Predictions',
}

for fname, title in plots.items():
    fpath = results_dir / fname
    if fpath.exists():
        fig, ax = plt.subplots(figsize=(12, 6))
        ax.imshow(mpimg.imread(str(fpath)))
        ax.axis('off')
        ax.set_title(title, fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f"/kaggle/working/outputs/{fname}", dpi=150)
        plt.show()
    else:
        print(f"Not found: {fname}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from ultralytics import YOLO
import glob, random

model = YOLO('runs/detect/water_bridge_s/weights/best.pt')

val_imgs = glob.glob("/kaggle/working/dataset/images/val/*.png")
sample   = random.sample(val_imgs, min(50, len(val_imgs)))

confidences = []
bridge_counts = []

results = model.predict(source=sample, conf=0.25, verbose=False)

for r in results:
    boxes = r.boxes
    if boxes is not None and len(boxes) > 0:
        confs = boxes.conf.cpu().numpy()
        confidences.extend(confs.tolist())
        bridge_counts.append(len(boxes))
    else:
        bridge_counts.append(0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Detection Analysis — Water Bridges', fontsize=13, fontweight='bold')

# Confidence distribution
axes[0].hist(confidences, bins=20, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Confidence Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Confidence Distribution')
axes[0].axvline(0.25, color='red', linestyle='--', label='threshold (0.25)')
axes[0].axvline(np.mean(confidences) if confidences else 0,
                color='green', linestyle='--',
                label=f'mean ({np.mean(confidences):.2f})')
axes[0].legend()

# Bridges per image
axes[1].hist(bridge_counts, bins=range(0, max(bridge_counts)+2),
             color='coral', edgecolor='white', align='left')
axes[1].set_xlabel('Bridges detected per image')
axes[1].set_ylabel('Number of images')
axes[1].set_title('Bridges Per Image Distribution')

plt.tight_layout()
plt.savefig("/kaggle/working/outputs/confidence_analysis.png", dpi=150)
plt.show()

print(f"\nTotal detections   : {len(confidences)}")
print(f"Avg confidence     : {np.mean(confidences):.3f}" if confidences else "No detections")
print(f"Avg bridges/image  : {np.mean(bridge_counts):.2f}")
print(f"Images with bridges: {sum(1 for c in bridge_counts if c > 0)}/{len(sample)}")

In [ ]:
import shutil
from pathlib import Path

# Kaggle auto-saves /kaggle/working/ when you click
# "Save Version" — but let's organize outputs cleanly

out = Path("/kaggle/working/final_outputs")
out.mkdir(exist_ok=True)

# Copy best model
shutil.copy(
    "runs/detect/water_bridge_s/weights/best.pt",
    out / "best.pt"
)

# Copy all plots
for f in Path("/kaggle/working/outputs").glob("*.png"):
    shutil.copy(f, out / f.name)

# Copy training results
for f in Path("runs/detect/water_bridge_s").glob("*.png"):
    shutil.copy(f, out / f.name)

print("Final outputs:")
for f in sorted(out.glob("*")):
    size = f.stat().st_size / 1e6
    print(f"  {f.name:40s} {size:.1f} MB")

In [ ]:
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from ultralytics import YOLO
import numpy as np

# ── CONFIG ───────────────────────────────────────
TEST_IMAGE = "/kaggle/working/dataset/images/val/v-100.png"  # ← change this path
CONF_THRESH = 0.25
MODEL_PATH  = "runs/detect/water_bridge_s/weights/best.pt"
# ─────────────────────────────────────────────────

model = YOLO(MODEL_PATH)

def detect_and_show(image_path, conf=0.25):
    image_path = Path(image_path)
    
    if not image_path.exists():
        print(f"Image not found: {image_path}")
        return
    
    # Run prediction
    results = model.predict(
        source=str(image_path),
        conf=conf,
        verbose=False
    )[0]

    # Load image for display
    img = cv2.imread(str(image_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]

    boxes      = results.boxes
    n_detected = len(boxes) if boxes is not None else 0

    # ── Plot ─────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    fig.suptitle(
        f"Bridge Over Water Detection — {image_path.name}\n"
        f"Bridges detected: {n_detected}",
        fontsize=14, fontweight='bold'
    )

    # Left — original image
    axes[0].imshow(img)
    axes[0].set_title("Original Image", fontsize=12)
    axes[0].axis('off')

    # Right — predictions
    axes[1].imshow(img)
    axes[1].set_title(f"Detections (conf ≥ {conf})", fontsize=12)
    axes[1].axis('off')

    if boxes is not None and n_detected > 0:
        for box in boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            conf_score      = float(box.conf[0].cpu().numpy())

            # Draw bounding box
            rect = patches.Rectangle(
                (x1, y1), x2-x1, y2-y1,
                linewidth=2,
                edgecolor='#00FF88',
                facecolor='none'
            )
            axes[1].add_patch(rect)

            # Confidence label
            axes[1].text(
                x1, y1 - 5,
                f"bridge {conf_score:.2f}",
                color='#00FF88',
                fontsize=9,
                fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2',
                          facecolor='black', alpha=0.6)
            )

        print(f"✓ {n_detected} bridge(s) detected over water")
        for i, box in enumerate(boxes):
            x1,y1,x2,y2 = box.xyxy[0].cpu().numpy()
            conf_score   = float(box.conf[0].cpu().numpy())
            w = x2-x1
            h = y2-y1
            print(f"  Bridge {i+1}: confidence={conf_score:.3f} | "
                  f"size={w:.0f}x{h:.0f}px | "
                  f"location=({x1:.0f},{y1:.0f})")
    else:
        axes[1].text(
            W//2, H//2,
            "No bridges detected",
            color='red', fontsize=14,
            ha='center', va='center',
            bbox=dict(facecolor='black', alpha=0.6)
        )
        print("No bridges detected — try lowering conf threshold")

    plt.tight_layout()

    # Save output
    out_path = f"/kaggle/working/outputs/detected_{image_path.name}"
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\nSaved: {out_path}")

# ── RUN ──────────────────────────────────────────
detect_and_show(TEST_IMAGE, conf=CONF_THRESH)